# ShqipAI Fine-Tuning: Phase 1 - Data Preparation

**Purpose**: Download and prepare FREE educational training data

**Hardware**: GPU P100 (free on Kaggle)

**Time**: ~30 minutes

**Output**: `educational_data.jsonl` file with ~50,000 training examples

## Step 1: Install Required Packages

In [ ]:
# Install packages (this takes ~2 minutes)
!pip install -q datasets transformers pandas tqdm

print("Packages installed successfully!")

## Step 2: Download FREE Educational Datasets

We'll combine multiple free sources for the best quality:

In [ ]:
from datasets import load_dataset, concatenate_datasets
import pandas as pd
from tqdm import tqdm

print("=" * 60)
print("DOWNLOADING FREE EDUCATIONAL DATASETS")
print("=" * 60)

all_data = []

### 2.1: OpenAssistant Conversations (Best Quality)

Real human-AI conversations with quality ratings.

In [ ]:
print("\n[1/4] Downloading OpenAssistant...")
try:
    oasst = load_dataset("OpenAssistant/oasst2")
    print(f"   Downloaded {len(oasst['train'])} conversations")
    
    # Filter for high-quality educational content
    def filter_educational(example):
        text = example.get('text', '').lower()
        keywords = ['explain', 'help', 'understand', 'how do', 'what is', 
                   'why', 'teach', 'learn', 'example', 'step']
        return any(kw in text for kw in keywords)
    
    educational = oasst['train'].filter(filter_educational)
    print(f"   Filtered to {len(educational)} educational conversations")
    
    for item in educational:
        all_data.append({
            'text': item['text'],
            'source': 'openassistant',
            'lang': item.get('lang', 'en'),
            'quality': 'high'
        })
        
except Exception as e:
    print(f"   Error: {e}")

### 2.2: UltraFeedback (High Quality Q&A)

Curated question-answer pairs with quality scores.

In [ ]:
print("\n[2/4] Downloading UltraFeedback...")
try:
    ultra = load_dataset("openbmb/UltraFeedback")
    print(f"   Downloaded {len(ultra['train'])} examples")
    
    for item in ultra['train'].select(range(min(10000, len(ultra['train'])))):
        # UltraFeedback has instruction-response pairs
        instruction = item.get('instruction', '')
        response = item.get('response', '')
        
        if instruction and response:
            all_data.append({
                'text': f"User: {instruction}\n\nTutor: {response}",
                'source': 'ultrafeedback',
                'lang': 'en',
                'quality': 'high'
            })
            
    print(f"   Added {min(10000, len(ultra['train']))} examples")
    
except Exception as e:
    print(f"   Error: {e}")

### 2.3: WizardLM (Math & Reasoning)

In [ ]:
print("\n[3/4] Downloading WizardLM...")
try:
    wizard = load_dataset("WizardLM/WizardLM_evol_instruct_V2_196k")
    print(f"   Downloaded {len(wizard['train'])} examples")
    
    # Focus on math and reasoning
    for item in wizard['train'].select(range(min(5000, len(wizard['train'])))):
        instruction = item.get('instruction', '')
        output = item.get('output', '')
        
        if instruction and output:
            all_data.append({
                'text': f"User: {instruction}\n\nTutor: {output}",
                'source': 'wizardlm',
                'lang': 'en',
                'quality': 'high'
            })
            
    print(f"   Added {min(5000, len(wizard['train']))} examples")
    
except Exception as e:
    print(f"   Error: {e}")

### 2.4: Add Multilingual Data (Albanian, etc.)

In [ ]:
print("\n[4/4] Downloading Multilingual Data...")
try:
    # Use OPUS-100 for parallel text (includes Albanian)
    opus = load_dataset("opus100", "en-sq")  # English-Albanian
    print(f"   Downloaded {len(opus['train'])} translation pairs")
    
    for item in opus['train'].select(range(min(2000, len(opus['train'])))):
        en_text = item['translation']['en']
        sq_text = item['translation']['sq']
        
        all_data.append({
            'text': f"English: {en_text}\nAlbanian: {sq_text}",
            'source': 'opus',
            'lang': 'sq',
            'quality': 'medium'
        })
        
    print(f"   Added {min(2000, len(opus['train']))} Albanian examples")
    
except Exception as e:
    print(f"   Error: {e}")

## Step 3: Format for Training

Convert all data to the format Gemma 4 expects.

In [ ]:
import json

print("\n" + "=" * 60)
print("FORMATTING DATA FOR TRAINING")
print("=" * 60)

def format_for_training(item):
    """Convert to Gemma 4 conversation format"""
    text = item['text']
    
    # Add system prompt for tutoring style
    system_prompt = "You are a patient, encouraging educational tutor. Explain concepts clearly, use examples, and check for understanding. Adapt your language to the student's level."
    
    # Format as conversation
    formatted = f"<|system|>\n{system_prompt}<|end|>\n{text}<|end|>"
    
    return {
        'text': formatted,
        'source': item['source'],
        'lang': item['lang'],
        'quality': item['quality']
    }

# Format all data
formatted_data = [format_for_training(item) for item in tqdm(all_data, desc="Formatting")]

print(f"\nTotal formatted examples: {len(formatted_data)}")

## Step 4: Save the Training Data

In [ ]:
print("\n" + "=" * 60)
print("SAVING TRAINING DATA")
print("=" * 60)

# Save as JSONL (one JSON object per line)
output_file = "educational_data.jsonl"

with open(output_file, 'w', encoding='utf-8') as f:
    for item in formatted_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print(f"\nSaved to: {output_file}")
print(f"Total examples: {len(formatted_data)}")

# Show statistics
sources = {}
langs = {}
for item in formatted_data:
    sources[item['source']] = sources.get(item['source'], 0) + 1
    langs[item['lang']] = langs.get(item['lang'], 0) + 1

print("\n--- Source Distribution ---")
for src, count in sources.items():
    print(f"  {src}: {count}")

print("\n--- Language Distribution ---")
for lang, count in langs.items():
    print(f"  {lang}: {count}")

## Step 5: Preview the Data

In [ ]:
# Show a few examples
print("\n" + "=" * 60)
print("SAMPLE TRAINING EXAMPLES")
print("=" * 60)

for i in range(3):
    print(f"\n--- Example {i+1} ---")
    print(formatted_data[i]['text'][:500] + "...")

## Done! Next Steps

1. **Download** the `educational_data.jsonl` file from the Output panel on the right
2. **Save** this file - you'll upload it to the TPU notebook
3. **Move to Phase 2**: Open the TPU training notebook